# 🛡️ CyberPulse: Automated Cyber Threat Intelligence & IOC Analyzer
### IBM Bootcamp Capstone Project Submission
**Candidate:** Swapnil Rathod  
**Domain:** Cybersecurity / Security Operations Center (SOC) Automation  
**Live Deployed Web App:** [https://swapnilrathod07.github.io/IBM-bootcamp-project/](https://swapnilrathod07.github.io/IBM-bootcamp-project/)  
**GitHub Repository:** [https://github.com/swapnilrathod07/IBM-bootcamp-project](https://github.com/swapnilrathod07/IBM-bootcamp-project)  
**Verification:** 187 / 187 Automated Unit Tests Passed (100% Pass Rate)


## 1. System Setup & Environment Configuration
This notebook executes the complete **CyberPulse** end-to-end security pipeline, demonstrating automated IOC ingestion, RFC 1918 loopback suppression, multi-source risk correlation, MITRE ATT&CK adversary profiling, 1-click defensive containment, and unit test execution.


In [1]:
import sys
import os
import json

# Ensure project root and CyberPulse modules are available in python path
ROOT_DIR = os.path.abspath(os.path.dirname(__file__) if '__file__' in locals() else '.')
CYBERPULSE_DIR = os.path.join(ROOT_DIR, 'CyberPulse')

for path in (ROOT_DIR, CYBERPULSE_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"[✓] System initialized. Python version: {sys.version.split()[0]}")
print(f"[✓] Project Root: {ROOT_DIR}")


[✓] System initialized. Python version: 3.13.0
[✓] Project Root: C:\Users\swapn\OneDrive\Desktop\My-Project\IBM-bootcamp-project


## 2. Threat Indicator Normalization, Defanging & Classification (`modules/detector.py`)
Automatically sanitizes defanged indicators (`hxxps[:]//`, `1[.]1[.]1[.]1`, `evil[.]com`) and classifies inputs into standard RFC types (IPv4, IPv6, MD5, SHA-256, Domain, URL).


In [2]:
from CyberPulse.modules.detector import detect_ioc_type, defang_ioc, refang_ioc

sample_inputs = [
    "hxxp[://]evil-phish[.]net/login?token=abc",
    "185[.]220[.]101[.]5",
    "2001:0db8:85a3:0000:0000:8a2e:0370:7334",
    "44d88612fea8a8f36de82e1278abb02f",
    "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f"
]

print(f"{'Input Indicator':<45} | {'Normalized':<35} | {'Detected Type'}")
print("-" * 100)
for inp in sample_inputs:
    detection = detect_ioc_type(inp)
    print(f"{inp:<45} | {detection['normalized']:<35} | {detection['type']} ({detection['type_label']})")


Input Indicator                               | Normalized                          | Detected Type
----------------------------------------------------------------------------------------------------
hxxp[://]evil-phish[.]net/login?token=abc     | http://evil-phish.net/login?token=abc | URL (Uniform Resource Locator)
185[.]220[.]101[.]5                           | 185.220.101.5                       | IPV4 (IPv4 Address)
2001:0db8:85a3:0000:0000:8a2e:0370:7334       | 2001:db8:85a3::8a2e:370:7334        | IPV6 (IPv6 Address)
44d88612fea8a8f36de82e1278abb02f             | 44d88612fea8a8f36de82e1278abb02f   | MD5 (MD5 Hash)
275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f | 275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f | SHA256 (SHA-256 Hash)


## 3. RFC 1918 & Localhost False-Positive Protection Engine
Naive scanners often flag internal assets (`127.0.0.1`, `192.168.1.1`, `http://localhost:5000/`) as malicious threats or leak private addresses to external threat APIs. CyberPulse intercepts private subnets and loopback addresses, classifying them safely with zero false alarms.


In [3]:
internal_samples = [
    "127.0.0.1",
    "http://127.0.0.1:5000/",
    "http://localhost:8501",
    "192.168.1.105",
    "10.0.0.1",
    "8.8.8.8"
]

print(f"{'Target Asset':<28} | {'Private IP?':<12} | {'Loopback?':<12} | {'Triage Status'}")
print("-" * 75)
for asset in internal_samples:
    d = detect_ioc_type(asset)
    status = "🛡️ SAFE INTERNAL ASSET (0 Risk)" if (d['is_private'] or d['is_loopback']) else "🌐 PUBLIC (Query Threat Intel)"
    print(f"{asset:<28} | {str(d['is_private']):<12} | {str(d['is_loopback']):<12} | {status}")


Target Asset                 | Private IP?  | Loopback?    | Triage Status
---------------------------------------------------------------------------
127.0.0.1                    | False        | True         | 🛡️ SAFE INTERNAL ASSET (0 Risk)
http://127.0.0.1:5000/       | False        | True         | 🛡️ SAFE INTERNAL ASSET (0 Risk)
http://localhost:8501        | False        | True         | 🛡️ SAFE INTERNAL ASSET (0 Risk)
192.168.1.105                | True         | False        | 🛡️ SAFE INTERNAL ASSET (0 Risk)
10.0.0.1                     | True         | False        | 🛡️ SAFE INTERNAL ASSET (0 Risk)
8.8.8.8                      | False        | False        | 🌐 PUBLIC (Query Threat Intel)


## 4. Multi-Source Threat Scoring & 14-Day Velocity Engine (`modules/scorer.py`)
Computes a unified 0–100 risk score with exponential half-life time-decay and generates a 14-day attack velocity trend line.


In [4]:
from CyberPulse.modules.scorer import calculate_risk_score, generate_attack_velocity

sample_ioc = "185.220.101.5"
vt_sample = {"positives": 68, "total": 88, "malicious": 68, "tags": ["ssh", "brute-force", "scanner"]}
abuse_sample = {"abuse_confidence_score": 100, "total_reports": 3420}
otx_sample = {"pulse_count": 45}

risk_result = calculate_risk_score(
    ioc=sample_ioc,
    ioc_type="IPV4",
    vt_data=vt_sample,
    abuse_data=abuse_sample,
    otx_data=otx_sample
)
velocity = generate_attack_velocity(
    ioc=sample_ioc,
    ioc_type="IPV4",
    vt_data=vt_sample,
    abuse_data=abuse_sample,
    otx_data=otx_sample
)

print(f"[!] Unified Threat Score: {risk_result['score']} / 100")
print(f"[!] Severity Level:       {risk_result['level']}")
print(f"[!] Velocity Trend:       {velocity['trend_direction']} ({velocity['trend_percentage']:+.1f}%)")
print(f"[!] Total 14-Day Events:  {velocity['total_14d_events']} hits (Peak: {velocity['peak_daily_events']} hits on {velocity['peak_date']})")
print(f"[!] Decay Mechanics:      {velocity['half_life_text']}")


[!] Unified Threat Score: 100 / 100
[!] Severity Level:       Malicious / Critical
[!] Velocity Trend:       surge (+167.5%)
[!] Total 14-Day Events:  2455 hits (Peak: 397 hits on 2026-10-23)
[!] Decay Mechanics:      t_1/2 = 14.0d (λ = 0.0495)


## 5. MITRE ATT&CK Adversary Profiling & Hacker Toolsets (`modules/attack_analyzer.py`)
Translates raw telemetry into an intuitive attack story and maps indicators to mapped adversary toolsets (*Mimikatz, PsExec, THC-Hydra*).


In [5]:
from CyberPulse.modules.attack_analyzer import analyze_attack_vector

attack = analyze_attack_vector(
    ioc=sample_ioc,
    ioc_type="IPV4",
    vt_data=vt_sample,
    abuse_data=abuse_sample,
    otx_data=otx_sample
)

tools = [t['name'] for t in attack.get('adversary_tools', [])]
mitre_techniques = [t['id'] + ': ' + t['name'] for t in attack.get('mitre_techniques', [])[:2]]

print("=" * 80)
print(f"ADVERSARY ATTACK PROFILE: {attack.get('name')}")
print("=" * 80)
print(f"Hacker Objective:  {attack.get('hacker_objective')}")
print(f"Blast Radius:      {attack.get('blast_radius')}")
print(f"MITRE Techniques:  {', '.join(mitre_techniques)}")
print(f"Mapped Toolsets:   {', '.join(tools)}")


ADVERSARY ATTACK PROFILE: SSH / RDP Brute Force
Hacker Objective:  Gain initial foothold and interactive administrative desktop/shell access via compromised user accounts.
Blast Radius:      Single-host takeover, privilege escalation to domain administrator, lateral movement throughout internal corporate subnet.
MITRE Techniques:  T1110.001: Password Guessing, T1110.003: Password Spraying
Mapped Toolsets:   THC-Hydra, Crowbar, Ncrack, Medusa


## 6. 1-Click Defensive Containment Rule Synthesizer (`modules/rule_gen.py`)
Generates actionable, copy-paste defense scripts across multiple platforms:
- **Windows Firewall**: PowerShell `New-NetFirewallRule`
- **Linux Firewall**: `iptables -A INPUT -s <IP> -j DROP`
- **Hunting Rules**: YARA for hashes & Sigma YAML for domain/URL hunting.


In [6]:
from CyberPulse.modules.rule_gen import generate_firewall_commands, generate_yara_rule

fw_rules = generate_firewall_commands("185.220.101.5")
yara_rule = generate_yara_rule("275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f", "WannaCry")

print("--- [1] POWERSHELL FIREWALL RULE ---")
print(fw_rules.get("powershell"))
print("\n--- [2] LINUX IPTABLES COMMAND ---")
print(fw_rules.get("iptables"))
print("\n--- [3] YARA DETECTION SIGNATURE ---")
print(yara_rule[:240] + "\n...")


--- [1] POWERSHELL FIREWALL RULE ---
New-NetFirewallRule -DisplayName "CyberPulse_Block_185_220_101_5" -Direction Outbound -RemoteAddress "185.220.101.5" -Action Block -Description "Automated perimeter block for high-risk IOC 185.220.101.5"

--- [2] LINUX IPTABLES COMMAND ---
# CyberPulse IOC Perimeter Containment
iptables -A OUTPUT -d 185.220.101.5 -j DROP
iptables -A INPUT -s 185.220.101.5 -j DROP

--- [3] YARA DETECTION SIGNATURE ---
rule CyberPulse_WannaCry_IOC_275a021b {
    meta:
        description = "Automated YARA rule generated by CyberPulse Threat Engine"
        author = "CyberPulse SOC Engine"
        date = "2026-10-23"
        threat_level = "CRITICAL"
    strings:
...


## 7. Web Security & SSL/TLS Defensive Headers Inspector (`modules/web_security.py`)
Inspects target hosts for valid SSL certificates, days until expiration, and essential defensive HTTP headers (`Strict-Transport-Security`, `Content-Security-Policy`, `X-Frame-Options`), producing a letter grade from **A+** to **F**.


In [7]:
from CyberPulse.modules.web_security import inspect_web_security

web_sec = inspect_web_security("https://github.com")

print(f"[✓] Target Domain:      {web_sec.get('domain')}")
print(f"[✓] SSL Certificate:    {web_sec.get('ssl_issuer')}")
print(f"[✓] Days to Expiration: {web_sec.get('days_to_expire')} days remaining")
print(f"[✓] Security Grade:     Grade {web_sec.get('grade')} ({web_sec.get('score')}/100)")
print(f"[✓] HSTS Status:        {web_sec.get('headers', {}).get('Strict-Transport-Security', {}).get('status')}")
print(f"[✓] CSP Status:         {web_sec.get('headers', {}).get('Content-Security-Policy', {}).get('status')}")


[✓] Target Domain:      github.com
[✓] SSL Certificate:    DigiCert Global G2 TLS RSA SHA256 2020 CA1
[✓] Days to Expiration: 284 days remaining
[✓] Security Grade:     Grade A+ (95/100)
[✓] HSTS Status:        Present (Strict Enforcement)
[✓] CSP Status:         Present (Robust Policy)


## 8. Verification & Automated Test Suite Execution
Executes all **187 unit tests** across the CyberPulse test framework to prove 100% reliability, zero crashes, and complete edge case coverage.


In [8]:
import unittest
import io

test_loader = unittest.TestLoader()
test_suite = test_loader.discover(start_dir=os.path.join(CYBERPULSE_DIR, 'tests'), pattern="test_*.py")

stream = io.StringIO()
runner = unittest.TextTestRunner(stream=stream, verbosity=1)
result = runner.run(test_suite)

print("=" * 80)
print(f"CYBERPULSE AUTOMATED TEST VERIFICATION SUMMARY")
print("=" * 80)
print(f"Total Tests Run: {result.testsRun}")
print(f"Failures:        {len(result.failures)}")
print(f"Errors:          {len(result.errors)}")
print(f"Success Rate:    {((result.testsRun - len(result.failures) - len(result.errors)) / result.testsRun) * 100:.1f}%")
print("=" * 80)
if result.wasSuccessful():
    print(">> [SUCCESS] ALL 187 UNIT TESTS PASSED WITH 100% ACCURACY! <<")


CYBERPULSE AUTOMATED TEST VERIFICATION SUMMARY
Total Tests Run: 187
Failures:        0
Errors:          0
Success Rate:    100.0%
>> [SUCCESS] ALL 187 UNIT TESTS PASSED WITH 100% ACCURACY! <<


## 9. Conclusion & Submission Links
- **Candidate:** Swapnil Rathod
- **Project Title:** CyberPulse — Automated Cyber Threat Intelligence & IOC Analyzer
- **Evaluation Track:** IBM Bootcamp Capstone Project
- **Live Deployment:** [https://swapnilrathod07.github.io/IBM-bootcamp-project/](https://swapnilrathod07.github.io/IBM-bootcamp-project/)
- **GitHub Repository:** [https://github.com/swapnilrathod07/IBM-bootcamp-project](https://github.com/swapnilrathod07/IBM-bootcamp-project)
